# Explorer un corpus de presse et comparer des tokenizers

Le corpus de ces travaux pratiques : tous les numéros de 1914 de deux quotidiens, *L'Humanité* (360 numéros) et *Le Figaro* (24 numéros), numérisés par la BnF. Le texte vient d'une reconnaissance de caractères (OCR) brute : il a les défauts d'un vrai corpus d'entreprise.

1. Charger le corpus et mesurer ses statistiques
2. Repérer et corriger les défauts de l'OCR, puis découper en passages
3. Comparer trois tokenizers, et estimer ce que coûte le traitement du corpus

## Données

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh french-press-1914
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/massive-intents/fr-test.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/massive-intents/en-test.parquet
!bash /tmp/ghsparse.sh dictionary-fr
%pip install -q chunspell

## 1. Charger le corpus

Chaque numéro est un fichier JSON, `french-press-1914/<journal>/1914/<AAAAMMJJ>/<AAAAMMJJ>.metadata.fulltext.json`, dont trois champs nous intéressent :

- `title` : le nom du journal, dans une liste (`contenu["title"][0]`)
- `date` : la date de parution, dans une liste (`contenu["date"][0]`)
- `contentAsText` : le texte, une chaîne par page, les lignes séparées par `\n`

Construisez un `pandas.DataFrame` nommé `numeros`, une ligne par numéro, avec les colonnes `newspaper`, `date` (au format date) et `text` (les pages jointes par `\n`). Affichez le nombre de numéros par journal et la distribution du nombre de mots par numéro.

**Documentation utile**

- [`Path.rglob`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.rglob)
- [`Path.read_text`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.read_text)
- [`json.loads`](https://docs.python.org/3/library/json.html#json.loads)
- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`pandas.to_datetime`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)
- [`Series.str.split`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.split.html)
- [`Series.describe`](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `Path("french-press-1914").rglob("*.json")` : tous les fichiers JSON, sous-dossiers compris (triés avec `sorted`)
- `chemin.read_text(encoding="utf8")`, puis `json.loads(...)`
- `contenu["title"][0]`, `contenu["date"][0]` et `"\n".join(contenu["contentAsText"])` : le journal, la date et le texte de chaque numéro
- `pd.DataFrame(donnees, columns=[...])` à partir d'une liste de tuples ; `pd.to_datetime` pour la colonne des dates
- `numeros.newspaper.value_counts()` ; `numeros.text.str.split().str.len().describe()` : la distribution du nombre de mots (`str.split()` sans argument découpe en mots)

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import json
from pathlib import Path

import pandas as pd

donnees = []
for chemin in sorted(Path("french-press-1914").rglob("*.json")):
  contenu = json.loads(chemin.read_text(encoding="utf8"))
  donnees.append(
    (..., ..., ...)
  )  # le journal, la date, et le texte (les pages jointes par "\n")
numeros = pd.DataFrame(donnees, columns=[..., ..., ...])
numeros["date"] = ...  # au format date

print(...)  # le nombre de numéros par journal
...  # la distribution du nombre de mots par numéro
```

### Solution

In [ ]:
import json
from pathlib import Path

import pandas as pd

donnees = []
for chemin in sorted(Path("french-press-1914").rglob("*.json")):
  contenu = json.loads(chemin.read_text(encoding="utf8"))
  donnees.append(
    (contenu["title"][0], contenu["date"][0], "\n".join(contenu["contentAsText"]))
  )
numeros = pd.DataFrame(donnees, columns=["newspaper", "date", "text"])
numeros["date"] = pd.to_datetime(numeros["date"])

print(numeros.newspaper.value_counts())
numeros.text.str.split().str.len().describe()

### Compter les mots

Comptez les formes de mots du corpus, en minuscules, avec l'expression régulière `MOT` ci-dessous et un `collections.Counter`. Affichez :

- le nombre total de mots et le nombre de formes différentes
- la part des formes qui n'apparaissent qu'une fois (les *hapax*)
- les 20 formes les plus fréquentes

**Documentation utile**

- [`re.Pattern.findall`](https://docs.python.org/3/library/re.html#re.Pattern.findall)
- [`collections.Counter`](https://docs.python.org/3/library/collections.html#collections.Counter)
- [`Counter.most_common`](https://docs.python.org/3/library/collections.html#collections.Counter.most_common)

In [ ]:
import re

MOT = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)?")

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `MOT.findall(texte)` : la liste de tous les mots d'un texte ; `.lower()` pour les minuscules
- `Counter(...)` se construit directement à partir d'un générateur de mots
- le total : `sum(comptes.values())` ; les hapax : les formes dont le compte vaut 1
- `comptes.most_common(20)` : les 20 formes les plus fréquentes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from collections import Counter

comptes = Counter(
  ... for texte in numeros.text for mot in MOT.findall(texte)
)  # chaque mot, en minuscules
total = sum(comptes.values())
hapax = sum(1 for nb in comptes.values() if ...)
print(f"{total:,} mots, {len(comptes):,} formes, {hapax / len(comptes):.0%} de hapax")
comptes.most_common(...)
```

#### Solution

In [ ]:
from collections import Counter

comptes = Counter(mot.lower() for texte in numeros.text for mot in MOT.findall(texte))
total = sum(comptes.values())
hapax = sum(1 for nb in comptes.values() if nb == 1)
print(f"{total:,} mots, {len(comptes):,} formes, {hapax / len(comptes):.0%} de hapax")
comptes.most_common(20)

### La loi de Zipf

Tracez la fréquence de chaque forme en fonction de son rang, en échelles logarithmiques, avec la droite de la loi de Zipf : une constante divisée par le rang, la constante étant la médiane du produit rang × fréquence entre les rangs 10 et 10 000 (ancrée sur le premier mot, la droite passerait sous les points). Les trois quarts des formes sont des hapax : affichez-en quinze, tirés au hasard avec `random.sample` après avoir fixé une graine avec `random.seed` (le même tirage à chaque exécution). À votre avis, qu'est-ce que c'est ?

**Documentation utile**

- [`matplotlib.pyplot.loglog`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.loglog.html)
- [`sorted`](https://docs.python.org/3/library/functions.html#sorted)
- [`numpy.arange`](https://numpy.org/doc/stable/reference/generated/numpy.arange.html)
- [`numpy.median`](https://numpy.org/doc/stable/reference/generated/numpy.median.html)
- [`random.seed`](https://docs.python.org/3/library/random.html#random.seed)
- [`random.sample`](https://docs.python.org/3/library/random.html#random.sample)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `sorted(comptes.values(), reverse=True)` : les fréquences, de la plus grande à la plus petite ; `np.arange(1, n + 1)` : les rangs, à partir de 1
- `plt.loglog(rangs, fréquences, label="...")` : les deux axes en échelle logarithmique ; `np.median(fréquences[9:10000] * rangs[9:10000])` : la constante, entre les rangs 10 et 10 000 (les indices commencent à 0) ; la droite de Zipf se trace avec `"--"` et `constante / rangs`
- `plt.xlabel("Rang")`, `plt.ylabel("Occurrences")`, `plt.legend()`, `plt.show()`
- les hapax : les formes de `comptes` dont le compte vaut 1 ; `random.seed(0)`, puis `random.sample(formes, 15)` : quinze au hasard, toujours les mêmes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import random

import matplotlib.pyplot as plt
import numpy as np

frequences = np.array(sorted(..., reverse=True))
rangs = np.arange(1, ... + 1)
constante = np.median(...)  # rang × fréquence, des rangs 10 à 10 000
plt.loglog(rangs, frequences, label="Presse de 1914")
plt.loglog(rangs, ..., "--", label="Loi de Zipf")  # la constante divisée par le rang
plt.xlabel("Rang")
plt.ylabel("Occurrences")
plt.legend()
plt.show()

random.seed(0)
random.sample([mot for mot, nb in comptes.items() if ...], 15)  # les hapax
```

#### Solution

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

frequences = np.array(sorted(comptes.values(), reverse=True))
rangs = np.arange(1, len(frequences) + 1)
plt.loglog(rangs, frequences, label="Presse de 1914")
constante = np.median(frequences[9:10000] * rangs[9:10000])
plt.loglog(rangs, constante / rangs, "--", label="Loi de Zipf")
plt.xlabel("Rang")
plt.ylabel("Occurrences")
plt.legend()
plt.show()

random.seed(0)
random.sample([mot for mot, nb in comptes.items() if nb == 1], 15)

# Surtout des erreurs d'OCR : des mots mal reconnus (« présldeht », « conjcerts »),
# collés (« en'lui ») ou réduits à des débris ; quelques noms propres abîmés
# (« scheiidémann ») et de vrais mots rares (« discrétionnaire »).

## 2. Les défauts de l'OCR

Les mots coupés en fin de ligne sont restés coupés : « con- tinuer ». L'expression régulière `COUPURE` ci-dessous cherche ces paires : un mot, un tiret, des espaces, puis un autre mot.

In [ ]:
COUPURE = re.compile(r"([^\W\d_]+)-\s+([^\W\d_]+)")

### Recoller les mots coupés

Recoller toutes les paires serait une erreur : « Sud- Ouest » doit rester en deux mots. Un correcteur orthographique dit si le mot recollé existe : la bibliothèque [`chunspell`](https://github.com/cdhigh/chunspell) donne accès à Hunspell, le moteur de LibreOffice, ici avec le dictionnaire français de Grammalecte. `Hunspell.spell(mot)` vaut `True` pour un mot connu. La casse compte : « Clemenceau » est connu, « clemenceau » ne l'est pas, et `connu` reçoit donc le mot tel quel. Un nom propre absent du dictionnaire, comme « Cail- laux », resterait coupé.

Le corpus peut servir de second dictionnaire : « Caillaux » y figure 1 851 fois. Mais une forme présente ailleurs ne suffit pas : « dela » y figure 52 fois, à cause d'une espace perdue, et près d'un tiers des paires ainsi ajoutées seraient fautives. On ne le consulte donc que pour ce qui ressemble à un nom propre : un premier morceau à majuscule initiale, un second en minuscules, cinq lettres au moins en tout, et une forme collée présente au moins deux fois dans `comptes`, qui a compté le texte d'avant le recollage.

Rangez dans `paires` toutes les correspondances de `COUPURE` dans le corpus. Écrivez une fonction `a_recoller(paire)` qui applique ces deux règles, puis `recolle(texte)` qui ne recolle que ces paires. Comptez les paires, celles recollées et celles laissées telles quelles, et affichez-en dix de chaque sorte, tirées au hasard avec `random.sample` (après `random.seed`) : que laisse la règle ? Recollez enfin le texte de chaque numéro. La fonction `connu` mémorise ses réponses : le corpus compte des millions de mots.

**Documentation utile**

- [`re.Pattern.finditer`](https://docs.python.org/3/library/re.html#re.Pattern.finditer)
- [`re.Match`](https://docs.python.org/3/library/re.html#paire-objects)
- [`re.Pattern.sub`](https://docs.python.org/3/library/re.html#re.Pattern.sub)
- [`Hunspell.spell`](https://github.com/cdhigh/chunspell#spelling)
- [`str.istitle`](https://docs.python.org/3/library/stdtypes.html#str.istitle)
- [`str.islower`](https://docs.python.org/3/library/stdtypes.html#str.islower)
- [`functools.cache`](https://docs.python.org/3/library/functools.html#functools.cache)
- [`Series.map`](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)
- [`random.sample`](https://docs.python.org/3/library/random.html#random.sample)

In [ ]:
from functools import cache

from hunspell import Hunspell  # le paquet chunspell

verifie = Hunspell("fr", hunspell_data_dir="dictionary-fr").spell


@cache
def connu(mot: str) -> bool:
  return verifie(mot)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `COUPURE.finditer(texte)` : un objet `re.Match` par correspondance ; `paire[0]` : toute la correspondance, `paire[1]` et `paire[2]` : ses deux morceaux
- `COUPURE.sub(remplace, texte)` : `remplace` reçoit chaque `re.Match` et renvoie le texte qui le remplace ; `paire[1] + paire[2]` recolle la paire, `paire[0]` la laisse telle quelle
- `connu(mot)` dit si un mot est connu du dictionnaire
- `premier.istitle()` : une majuscule initiale, puis des minuscules ; `second.islower()` : tout en minuscules
- `comptes[colle.lower()]` : les occurrences de la forme collée (`comptes` est en minuscules), 0 si elle est absente
- les paires recollées : `[paire[0] for paire in paires if a_recoller(paire)]`, et de même celles laissées ; `random.seed(0)`, puis `random.sample(recollees, 10)` : dix au hasard, toujours les mêmes
- `numeros.text.map(recolle)` applique la fonction à chaque texte

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
paires = [paire for texte in numeros.text for paire in COUPURE.finditer(...)]


def a_recoller(paire: re.Match[str]) -> bool:
  premier, second = paire[1], paire[2]
  colle = premier + second
  nom_propre = ... and ... and len(colle) >= 5  # majuscule initiale, puis minuscules
  return connu(colle) or (nom_propre and ...)  # au moins deux fois dans le corpus


def recolle(texte: str) -> str:
  def remplace(paire: re.Match[str]) -> str:
    return (
      ... if a_recoller(paire) else ...
    )  # le mot recollé, ou la correspondance telle quelle

  return COUPURE.sub(remplace, texte)


recollees = [paire[0] for paire in paires if ...]  # les paires à recoller
laissees = [paire[0] for paire in paires if ...]  # celles laissées telles quelles
print(f"{len(paires)} paires, {len(recollees)} recollées, {len(laissees)} laissées")
random.seed(0)
print(random.sample(recollees, ...))
print(random.sample(laissees, ...))
numeros["text"] = numeros.text.map(...)
```

#### Solution

In [ ]:
paires = [paire for texte in numeros.text for paire in COUPURE.finditer(texte)]


def a_recoller(paire: re.Match[str]) -> bool:
  premier, second = paire[1], paire[2]
  colle = premier + second
  nom_propre = premier.istitle() and second.islower() and len(colle) >= 5
  return connu(colle) or (nom_propre and comptes[colle.lower()] >= 2)


def recolle(texte: str) -> str:
  def remplace(paire: re.Match[str]) -> str:
    return paire[1] + paire[2] if a_recoller(paire) else paire[0]

  return COUPURE.sub(remplace, texte)


recollees = [paire[0] for paire in paires if a_recoller(paire)]
laissees = [paire[0] for paire in paires if not a_recoller(paire)]
print(f"{len(paires)} paires, {len(recollees)} recollées, {len(laissees)} laissées")
random.seed(0)
print(random.sample(recollees, 10))
print(random.sample(laissees, 10))
numeros["text"] = numeros.text.map(recolle)

# Recollées : des mots (« étincel- lement ») et des noms propres (« Gam- betta »).
# Laissées : du bruit d'OCR (« potir- Cfeeiboi »), des mots composés (« Autriche-
# Hongrie », « Saint- Lazare »), des espaces perdues (« il- fallait ») et des mots
# coupés mais mal reconnus (« Fon- tienav »), que la règle ne peut pas recoller.

### Le taux de bruit

Le **taux de bruit** d'un texte : la part de ses mots (de plus d'une lettre) que le dictionnaire ne connaît pas. Écrivez une fonction `bruit(texte)`, calculez-la pour chaque numéro et comparez les deux journaux. Affichez les 20 formes inconnues de plus de cinq lettres les plus fréquentes du corpus (les plus courtes sont surtout des débris : « da », « tre ») : erreurs d'OCR, noms propres, orthographe ancienne ?

Beaucoup sont des moitiés de mots coupés par une espace, sans tiret : « gou vernement », « so cialiste ». La version publiée des passages les recolle aussi, quand l'un des deux morceaux est inconnu et le mot recollé connu.

**Documentation utile**

- [`Series.map`](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)
- [`DataFrame.groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [`Counter.most_common`](https://docs.python.org/3/library/collections.html#collections.Counter.most_common)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- les mots d'un texte de plus d'une lettre : `[mot for mot in MOT.findall(texte) if len(mot) > 1]`
- le taux de bruit : la part de ces mots que `connu` ne connaît pas ; `max(len(mots), 1)` évite une division par zéro
- `numeros.text.map(bruit)`, puis `numeros.groupby("newspaper").noise.describe()` : les statistiques par journal
- `Counter(...).most_common(20)` sur les formes inconnues de plus de cinq lettres (`len(mot) > 5`), en minuscules

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def mots(texte: str) -> list[str]:
  return [mot for mot in MOT.findall(texte) if ...]  # les mots de plus d'une lettre


def bruit(texte: str) -> float:
  trouves = mots(texte)
  return sum(... for mot in trouves) / max(len(trouves), 1)  # la part des mots inconnus


numeros["noise"] = numeros.text.map(...)
print(numeros.groupby(...).noise.describe())
Counter(
  mot.lower() for texte in numeros.text for mot in mots(texte) if ...
).most_common(20)
```

#### Solution

In [ ]:
def mots(texte: str) -> list[str]:
  return [mot for mot in MOT.findall(texte) if len(mot) > 1]


def bruit(texte: str) -> float:
  trouves = mots(texte)
  return sum(not connu(mot) for mot in trouves) / max(len(trouves), 1)


numeros["noise"] = numeros.text.map(bruit)
print(numeros.groupby("newspaper").noise.describe())
Counter(
  mot.lower()
  for texte in numeros.text
  for mot in mots(texte)
  if len(mot) > 5 and not connu(mot)
).most_common(20)

# Des noms propres (Caillaux, Barthou), inconnus du dictionnaire même recollés, des
# capitales sans accent (l'Etat, THEATRE) et des moitiés de mots (vernement).

### Découper en passages

Un numéro entier est trop long pour un modèle, et mêle des dizaines d'articles : on le découpe en **passages**. Écrivez une fonction `passages(texte)` qui accumule les lignes d'un numéro jusqu'à atteindre au moins 100 mots, et coupe à la première ligne qui finit une phrase (par `.`, `!`, `?` ou `»`). Combien de passages obtenez-vous, et de combien de mots ?

Comparez ensuite avec `passages.parquet`, la version publiée pour les TPs suivants : elle recolle les paires et mesure le bruit avec la liste de mots de `wordfreq`, plus tolérante que notre dictionnaire (18 980 paires recollées), découpe aussi en phrases les lignes de plus de 250 mots, et écarte les passages dont le bruit dépasse 0,15 et les doublons. Combien de passages, de quelle longueur ? Comparez les deux distributions du nombre de mots : d'où vient l'écart ? Lisez trois passages publiés au hasard.

**Documentation utile**

- [`str.split`](https://docs.python.org/3/library/stdtypes.html#str.split)
- [`str.endswith`](https://docs.python.org/3/library/stdtypes.html#str.endswith)
- [`pandas.read_parquet`](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)
- [`Series.sample`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sample.html)
- [`Series.describe`](https://pandas.pydata.org/docs/reference/api/pandas.Series.describe.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `texte.split("\n")` : les lignes ; `" ".join(ligne.split())` normalise les espaces d\'une ligne ; `ligne.split()` sans argument : les mots
- `FIN = re.compile(r"[.!?»]\s*$")` et `FIN.search(ligne)` : la ligne finit une phrase (ou `ligne.endswith((".", "!", "?", "»"))`)
- accumulez les lignes dans `courant` et leurs mots dans `n` ; à 100 mots au moins et une fin de phrase, ajoutez `" ".join(courant)` à la liste des passages et repartez de zéro
- le nombre de mots de vos passages : `pd.Series([len(passage.split()) for passage in vos_passages]).describe()`, à comparer à `publies.n_words.describe()` ; pour l'écart, la fonction ne coupe qu'en fin de ligne : combien de mots compte une ligne de l'OCR ?
- `pd.read_parquet("passages.parquet")`, `publies.newspaper.value_counts()`, `publies.n_words.describe()`
- `publies.text.sample(3, random_state=0)` : trois passages au hasard, toujours les mêmes

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
FIN = re.compile(r"[.!?»]\s*$")


def passages(texte: str) -> list[str]:
  resultat, courant, n = [], [], 0
  for ligne in texte.split("\n"):
    ligne = " ".join(ligne.split())
    if not ligne:
      continue
    courant.append(ligne)
    n += len(ligne.split())
    if n >= ... and FIN.search(ligne):
      resultat.append(" ".join(courant))
      courant, n = ..., ...  # on repart de zéro
  return resultat


vos_passages = [passage for texte in numeros.text for passage in passages(texte)]
print(len(vos_passages))
# le nombre de mots de vos passages
print(pd.Series([... for passage in vos_passages]).describe())

publies = pd.read_parquet(...)
print(publies.newspaper.value_counts())
print(publies.n_words.describe())
for texte in publies.text.sample(3, random_state=0):
  print(texte, end="\n\n")
```

#### Solution

In [ ]:
FIN = re.compile(r"[.!?»]\s*$")


def passages(texte: str) -> list[str]:
  resultat, courant, n = [], [], 0
  for ligne in texte.split("\n"):
    ligne = " ".join(ligne.split())
    if not ligne:
      continue
    courant.append(ligne)
    n += len(ligne.split())
    if n >= 100 and FIN.search(ligne):
      resultat.append(" ".join(courant))
      courant, n = [], 0
  return resultat


vos_passages = [passage for texte in numeros.text for passage in passages(texte)]
print(len(vos_passages))
print(pd.Series([len(passage.split()) for passage in vos_passages]).describe())

publies = pd.read_parquet("passages.parquet")
print(publies.newspaper.value_counts())
print(publies.n_words.describe())
for texte in publies.text.sample(3, random_state=0):
  print(texte, end="\n\n")

# Vos passages : 334 mots en médiane, 9 003 au plus, contre 118 et 573 pour les
# passages publiés. La fonction ne coupe qu'en fin de ligne, et certaines lignes de
# l'OCR sont des paragraphes entiers : une sur dix dépasse 250 mots, jusqu'à 1 664.
# Une ligne de 1 000 mots n'est pas un passage : d'où la version publiée, qui coupe
# ces lignes en phrases.

## 3. Comparer des tokenizers

Chargez trois tokenizers avec `transformers.AutoTokenizer.from_pretrained` (le tokenizer seul se télécharge en quelques secondes) :

- `google-bert/bert-base-multilingual-cased` (WordPiece)
- `Qwen/Qwen3-Embedding-0.6B` (BPE sur les octets ; le modèle d'*embeddings* des TPs suivants)
- `FacebookAI/xlm-roberta-base` (Unigram)

Pour chacun, affichez les tokens et le nombre de tokens par mot des textes de `exemples`. `convert_ids_to_tokens` écrit les tokens comme le vocabulaire, marqueurs compris : `##` (WordPiece) dit qu'un token continue un mot, `▁` (XLM-R) qu'il en commence un. Qu'est-ce qui coûte le plus de tokens ?

Quatre phrases ne suffisent pas pour comparer deux langues. `fr-test.parquet` et `en-test.parquet`, téléchargés plus haut, tiennent les mêmes 2 974 demandes à un assistant vocal, en français et en anglais (le jeu de données MASSIVE, que reprennent les TPs de classification d'intentions). Pour chaque tokenizer, mesurez dans chaque langue le nombre de mots et de tokens par mot, puis le rapport des tokens du français à ceux de l'anglais : à sens égal, combien de tokens de plus le français coûte-t-il, et pourquoi ?

**Documentation utile**

- [`AutoTokenizer.from_pretrained`](https://huggingface.co/docs/transformers/model_doc/auto#transformers.AutoTokenizer.from_pretrained)
- [`tokenizer(...)`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.__call__)
- [`tokenizer.convert_ids_to_tokens`](https://huggingface.co/docs/transformers/internal/tokenization_utils#transformers.PreTrainedTokenizerBase.convert_ids_to_tokens)
- [`pandas.read_parquet`](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html)
- [Les algorithmes de tokenisation](https://huggingface.co/docs/transformers/tokenizer_summary)

In [ ]:
exemples = {
  "français": "Le conseil municipal a voté hier soir le budget de la nouvelle médiathèque.",
  "anglais": "The city council voted on the budget for the new media library last night.",
  "nom propre": "Clemenceau et Poincaré",
  "OCR": "Où .av b; aifn'wmant Mme Gueydan, it. o ta î - iu ei est arrivée a la barre",
}

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `AutoTokenizer.from_pretrained(nom)` ; rangez les trois dans un dictionnaire `tokenizers` (nom → tokenizer) : un exercice suivant s'en sert
- `tokenizer(texte, add_special_tokens=False).input_ids` : les identifiants des tokens, sans les tokens spéciaux (`[CLS]`, `<s>`…) qu'on ne veut pas compter
- `tokenizer.convert_ids_to_tokens(ids)` : les tokens d'une liste d'identifiants. Qwen3 découpe des octets, pas des caractères : `Ġ` y note l'espace devant un mot, et une lettre accentuée, deux octets en UTF-8, s'y écrit en deux caractères (`Ã©` pour « é »)
- tokens par mot : `len(ids) / len(texte.split())`
- `pd.read_parquet("fr-test.parquet").text` : les demandes en français ; `textes.str.split().str.len().sum()` : leur nombre de mots ; leur nombre de tokens : la somme des `len(ids)` de chaque demande

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
from transformers import AutoTokenizer

tokenizers = {
  nom: AutoTokenizer.from_pretrained(...)
  for nom in (
    "google-bert/bert-base-multilingual-cased",
    "Qwen/Qwen3-Embedding-0.6B",
    "FacebookAI/xlm-roberta-base",
  )
}
for nom, tokenizer in tokenizers.items():
  print(nom)
  for etiquette, texte in exemples.items():
    ids = tokenizer(texte, add_special_tokens=False).input_ids
    tokens = tokenizer.convert_ids_to_tokens(...)
    print(f"  {etiquette:10} {len(ids) / len(texte.split()):.2f} tokens/mot {tokens}")

massive = {
  "fr": pd.read_parquet("fr-test.parquet").text,
  "en": pd.read_parquet(...).text,
}
nb_mots = {langue: ... for langue, textes in massive.items()}  # le nombre de mots
print(f"{nb_mots['fr']:,} mots en français, {nb_mots['en']:,} en anglais")
for nom, tokenizer in tokenizers.items():
  nb_tokens = {
    langue: sum(
      len(tokenizer(texte, add_special_tokens=False).input_ids) for texte in textes
    )
    for langue, textes in massive.items()
  }
  print(
    f"{nom}: {nb_tokens['fr'] / nb_mots['fr']:.2f} tokens/mot en français, "
    f"{nb_tokens['en'] / nb_mots['en']:.2f} en anglais, "
    f"{... / ...:.2f} fois plus de tokens en français"
  )
```

### Solution

In [ ]:
from transformers import AutoTokenizer

tokenizers = {
  nom: AutoTokenizer.from_pretrained(nom)
  for nom in (
    "google-bert/bert-base-multilingual-cased",
    "Qwen/Qwen3-Embedding-0.6B",
    "FacebookAI/xlm-roberta-base",
  )
}
for nom, tokenizer in tokenizers.items():
  print(nom)
  for etiquette, texte in exemples.items():
    ids = tokenizer(texte, add_special_tokens=False).input_ids
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f"  {etiquette:10} {len(ids) / len(texte.split()):.2f} tokens/mot {tokens}")

massive = {
  "fr": pd.read_parquet("fr-test.parquet").text,
  "en": pd.read_parquet("en-test.parquet").text,
}
nb_mots = {
  langue: textes.str.split().str.len().sum() for langue, textes in massive.items()
}
print(f"{nb_mots['fr']:,} mots en français, {nb_mots['en']:,} en anglais")
for nom, tokenizer in tokenizers.items():
  nb_tokens = {
    langue: sum(
      len(tokenizer(texte, add_special_tokens=False).input_ids) for texte in textes
    )
    for langue, textes in massive.items()
  }
  print(
    f"{nom}: {nb_tokens['fr'] / nb_mots['fr']:.2f} tokens/mot en français, "
    f"{nb_tokens['en'] / nb_mots['en']:.2f} en anglais, "
    f"{nb_tokens['fr'] / nb_tokens['en']:.2f} fois plus de tokens en français"
  )

# Le nom propre coûte le plus par mot (3 tokens par mot pour Qwen3 et XLM-R : quelques
# lettres chacun), puis le texte bruité de l'OCR (ses morceaux de mots ne sont pas dans
# les vocabulaires).
# Sur MASSIVE, le français prend 8 % de mots de plus que l'anglais, et plus de tokens
# par mot avec les trois tokenizers (1,54 contre 1,08 pour Qwen3) : à sens égal, 1,32
# fois plus de tokens pour BERT, 1,35 pour XLM-R, 1,54 pour Qwen3. Les vocabulaires,
# appris sur des corpus où l'anglais domine, découpent davantage le français. XLM-R,
# presque à égalité sur la phrase de `exemples`, ne l'est plus ici : une phrase ne
# prouve rien. Ces chiffres ne sont pas ceux de la figure du cours, mesurée sur
# d'autres phrases.

### Le coût du corpus

Combien de tokens (pour le tokenizer de Qwen3) compte le corpus de passages publié ? Estimez-le sur un échantillon de 2 000 passages. À 0,10 € par million de tokens d'entrée (l'ordre de grandeur d'un petit modèle par API), combien coûte un passage du corpus entier dans un LLM, rien qu'en entrée ?

Un encodeur limité à 512 tokens (BERT, XLM-R) ne voit que les 512 premiers d'un texte plus long : la suite est **tronquée**, souvent sans avertissement. Pour chacun des trois tokenizers, quelle part des passages de l'échantillon dépasse 512 tokens ? Et parmi 500 de vos passages (`vos_passages`), tirés au hasard ?

**Documentation utile**

- [`Series.sample`](https://pandas.pydata.org/docs/reference/api/pandas.Series.sample.html)
- [`tokenizer(...)`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.TokenizersBackend.__call__)
- [`numpy.mean`](https://numpy.org/doc/stable/reference/generated/numpy.mean.html)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `tokenizers["Qwen/Qwen3-Embedding-0.6B"]` : le tokenizer de Qwen3, de l'exercice précédent
- `publies.text.sample(2000, random_state=0)` : 2 000 passages au hasard, toujours les mêmes
- `len(tokenizer(texte, add_special_tokens=False).input_ids)` : le nombre de tokens d'un passage, sans les tokens spéciaux (le tokenizer de Qwen3 ajoute `<|endoftext|>` à chaque texte) ; `np.mean` en fait la moyenne
- `len(tokenizer(texte).input_ids)` : la longueur que voit le modèle, tokens spéciaux compris (`[CLS]` et `[SEP]` occupent deux des 512 places de BERT)
- le total : la moyenne × le nombre de passages (`len(publies)`) ; le coût : le total en millions de tokens × 0,10 €
- `pd.Series(vos_passages).sample(500, random_state=0)` : 500 de vos passages, toujours les mêmes
- `np.array(tailles) > 512` : un tableau de booléens, dont `np.mean` donne la part de `True`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
tokenizer = tokenizers["Qwen/Qwen3-Embedding-0.6B"]
echantillon = publies.text.sample(..., random_state=0)
tokens_par_passage = np.mean(
  [len(tokenizer(texte, add_special_tokens=False).input_ids) for texte in ...]
)
tokens_total = tokens_par_passage * ...
print(
  f"{tokens_par_passage:.0f} tokens par passage, {tokens_total / 1e6:.1f} millions au total"
)
print(f"{tokens_total / 1e6 * ...:.2f} € pour un passage du corpus")

echantillon_vos = pd.Series(vos_passages).sample(..., random_state=0)
for nom, tokenizer in tokenizers.items():
  tailles_publies = np.array([len(tokenizer(texte).input_ids) for texte in echantillon])
  tailles_vos = np.array([... for texte in echantillon_vos])
  print(
    f"{nom}: {np.mean(tailles_publies > ...):.2%} des passages publiés, "
    f"{np.mean(tailles_vos > ...):.0%} des vôtres au-delà de 512 tokens"
  )
```

#### Solution

In [ ]:
tokenizer = tokenizers["Qwen/Qwen3-Embedding-0.6B"]
echantillon = publies.text.sample(2000, random_state=0)
tokens_par_passage = np.mean(
  [len(tokenizer(texte, add_special_tokens=False).input_ids) for texte in echantillon]
)
tokens_total = tokens_par_passage * len(publies)
print(
  f"{tokens_par_passage:.0f} tokens par passage, {tokens_total / 1e6:.1f} millions au total"
)
print(f"{tokens_total / 1e6 * 0.10:.2f} € pour un passage du corpus")

echantillon_vos = pd.Series(vos_passages).sample(500, random_state=0)
for nom, tokenizer in tokenizers.items():
  tailles_publies = np.array([len(tokenizer(texte).input_ids) for texte in echantillon])
  tailles_vos = np.array([len(tokenizer(texte).input_ids) for texte in echantillon_vos])
  print(
    f"{nom}: {np.mean(tailles_publies > 512):.2%} des passages publiés, "
    f"{np.mean(tailles_vos > 512):.0%} des vôtres au-delà de 512 tokens"
  )

# Moins de 1 % des passages publiés dépassent 512 tokens (0,10 % avec BERT, 0,25 %
# avec XLM-R, 0,75 % avec Qwen3), mais six sur dix des vôtres (58 à 61 %) : un
# encodeur limité à 512 tokens n'en lirait que le début. Les tokenizers de BERT et de
# XLM-R le signalent d'un avertissement, mais ne coupent rien ; un modèle
# d'embeddings chargé par sentence-transformers tronque sans rien dire. D'où des
# passages courts. Le tokenizer de Qwen3 annonce `model_max_length` = 131 072, mais sa
# fiche donne 32 768 tokens (32k) au modèle : c'est la fiche qui fait foi.